# Copa 2026 — partidas isoladas (10 mil simulações por jogo)

Este notebook calcula probabilidades dos **três desfechos dos 90 minutos** de cada partida isolada, por Poisson e Monte Carlo, usando **10.000 sorteios por confronto**. Não monta grupos, chaveamento ou campeão. A versão histórica do código fornecido pelo pesquisador chamava 5.000 sorteios em um ponto; esta versão utiliza 10.000 conforme a metodologia pretendida.

As probabilidades usam os **scores agregados corrigidos** e podem diferir das probabilidades históricas salvas antes da revisão dos nomes. Os resultados finais de alguns jogos eliminatórios incluem prorrogação e não podem validar previsões exclusivas dos 90 minutos sem recodificação.


In [ ]:
!git clone https://github.com/warner03/Previs-o_de_Resultados_no_Futebol-via-Simula-o-de-Monte-Carlo_Football-Manager_Base-de-Dados.git
%cd /content/Previs-o_de_Resultados_no_Futebol-via-Simula-o-de-Monte-Carlo_Football-Manager_Base-de-Dados/copa-do-mundo
%pip -q install numpy==2.3.5


In [ ]:
import numpy as np, csv, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()/"codigos"))
from simulacao_campeonato_completo_copa_2026 import load_model, load_csv, REAL_NAME, Tournament, annex_lookup
grupos, rankings, scores, goleiros, atributos = load_model(5)
motor = Tournament(np.random.default_rng(20261109), grupos, rankings, scores, goleiros, annex_lookup())
rng = np.random.default_rng(20261109)
N_POR_PARTIDA = 10000

def estimar_confronto(a, b, n=N_POR_PARTIDA):
    la, lb = motor.lambdas(a, b)
    gols_a = rng.poisson(la, size=n)
    gols_b = rng.poisson(lb, size=n)
    return float(np.mean(gols_a > gols_b)), float(np.mean(gols_a == gols_b)), float(np.mean(gols_a < gols_b))

print("Brasil x França:", estimar_confronto("Brasil","França"))


In [ ]:
# Todos os 104 confrontos da planilha: cada um recebe 10.000 sorteios independentes
jogos = load_csv("dados/resultados_copa_2026.csv")
linhas=[]
for jogo in jogos:
    a=REAL_NAME[jogo["Mandante"]]; b=REAL_NAME[jogo["Visitante"]]
    pa,pe,pb=estimar_confronto(a,b)
    gols_a=int(jogo["Gols_Mandante"]); gols_b=int(jogo["Gols_Visitante"])
    y=(1,0,0) if gols_a>gols_b else ((0,1,0) if gols_a==gols_b else (0,0,1))
    probs=(pa,pe,pb)
    acertou=int(int(np.argmax(probs)) == y.index(1))
    brier=sum((p-o)**2 for p,o in zip(probs,y))
    linhas.append({"fase":jogo["Fase"],"primeira_selecao":a,"segunda_selecao":b,
                   "p_vitoria_primeira":pa,"p_empate":pe,"p_vitoria_segunda":pb,
                   "acerto_nominal":acertou,"brier_multiclasse":brier,
                   "comparavel_90min":jogo["Fase"]=="Grupos"})
grupos_real=[x for x in linhas if x["fase"]=="Grupos"]
print("Fase de grupos:",sum(x["acerto_nominal"] for x in grupos_real),"/",len(grupos_real),
      "Brier:",round(sum(x["brier_multiclasse"] for x in grupos_real)/len(grupos_real),4))
print("AVISO: resultados agregados das 104 partidas podem misturar gols de 90 e 120 minutos.")


In [ ]:
saida=Path("resultados/gerados/partidas_isoladas_10mil_copa_2026.csv")
saida.parent.mkdir(exist_ok=True,parents=True)
with saida.open("w",encoding="utf-8-sig",newline="") as arquivo:
    escritor=csv.DictWriter(arquivo,fieldnames=list(linhas[0]))
    escritor.writeheader()
    escritor.writerows(linhas)
print("Arquivo de resultados:",saida)
